# IT3170 preprocessing v3 — evidence-first Qwen2.5-VL-7B
Graph: nodes → edges → attributes, each with local evidence. State trees use boards/cells, not generic labels.
Native vectors and CV boards are proposals, not verified facts. All new results remain in Review Queue.
The default is the six-crop pilot on five difficult pages. Full inference is explicitly configured below, not automatic.


In [ ]:
import sys, subprocess, json, hashlib, zipfile, shutil, os, time
from pathlib import Path
from google.colab import drive,files
assert (3,10)<=sys.version_info[:2]<=(3,13)
subprocess.run(['nvidia-smi'],check=True)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/BKQuiz_region_v3');BASE.mkdir(parents=True,exist_ok=True)
uploaded=files.upload() # Select region_v3_input.zip
if 'region_v3_input.zip' not in uploaded:raise ValueError('Select the v3 input ZIP')
archive=BASE/('input_'+hashlib.sha256(uploaded['region_v3_input.zip']).hexdigest()[:12]+'.zip')
if not archive.exists():archive.write_bytes(uploaded['region_v3_input.zip'])
del uploaded
with zipfile.ZipFile(archive) as z:
    RUN_HASH=hashlib.sha256(z.read('manifest.json')).hexdigest()
    WORK=Path('/content')/('it3170_v3_'+RUN_HASH[:12]);WORK.mkdir(exist_ok=True)
    for info in z.infolist():
        if not (WORK/info.filename).resolve().is_relative_to(WORK.resolve()):raise ValueError('Unsafe ZIP member')
    z.extractall(WORK)
sys.path.insert(0,str(WORK))
from contracts import check_package
MANIFEST=check_package(WORK)
OUT=BASE/('results_'+RUN_HASH[:12]);OUT.mkdir(exist_ok=True)
os.environ['REGION_V3_CACHE']=str(BASE/'phase_cache')
NEEDS_PADDLE=any(r['engine'] in ['formula','table'] and r['execution']=='inference' and r['id']+'.json' not in MANIFEST['seed_sha256'] for r in MANIFEST['requests'])
print('Paddle inference needed:',NEEDS_PADDLE)
print('Pilot pages:',MANIFEST['pilot_pages'])
print('Output:',OUT)


## Environment
Restore the previous Python-matched venv archives. Paddle installation is skipped when all specialist outputs are seeded.
Model weights are separate from venv archives. Qwen uses NF4, FP16 compute and batch 1 on a CUDA GPU such as T4.


In [ ]:
PADDLE_ENV=Path('/content/it3170_paddle_env');VLM_ENV=Path('/content/it3170_vlm_env')
ENV_CACHE=Path('/content/drive/MyDrive/BKQuiz_preprocessing')
tag=f'py{sys.version_info.major}{sys.version_info.minor}'
setup_start=time.perf_counter()
def runnable(env):
    py=env/'bin/python'
    if not py.exists():return False
    return subprocess.run([str(py),'-c','import pip'],capture_output=True).returncode==0
for env in ([PADDLE_ENV] if NEEDS_PADDLE else [])+[VLM_ENV]:
    if runnable(env):continue
    if env.exists() or env.is_symlink():env.rename(env.with_name(env.name+'_broken_'+str(time.time_ns())))
    archive=ENV_CACHE/f'{env.name}_{tag}_v1.tar.gz'
    if archive.exists():
        local=Path('/content')/archive.name;shutil.copy2(archive,local)
        listing=subprocess.check_output(['tar','-tzf',str(local)],text=True).splitlines()
        if any(Path(n).is_absolute() or '..' in Path(n).parts or Path(n).parts[0]!=env.name for n in listing):raise ValueError('Unsafe venv archive')
        subprocess.run(['tar','-xzf',str(local),'-C','/content'],check=True)
    if not runnable(env):
        if env.exists():env.rename(env.with_name(env.name+'_incompatible_'+str(time.time_ns())))
        subprocess.run([sys.executable,'-m','venv','--without-pip',str(env)],check=True)
        subprocess.run([sys.executable,'-m','pip','--python',str(env/'bin/python'),'install','pip==25.2'],check=True)
PADDLE_PY=str(PADDLE_ENV/'bin/python');VLM_PY=str(VLM_ENV/'bin/python')
def versions_match(py,expected):
    source="import importlib.metadata as m,json;print(json.dumps({k:m.version(k).split('+')[0] for k in "+repr(list(expected))+"}))"
    p=subprocess.run([py,'-c',source],capture_output=True,text=True)
    return p.returncode==0 and json.loads(p.stdout)==expected
paddle_versions={'paddleocr':'3.7.0','paddlex':'3.7.2','paddlepaddle-gpu':'3.2.2'}
if NEEDS_PADDLE and not versions_match(PADDLE_PY,paddle_versions):
    subprocess.run([PADDLE_PY,'-m','pip','install','paddlepaddle-gpu==3.2.2','--index-url','https://www.paddlepaddle.org.cn/packages/stable/cu118/'],check=True)
    subprocess.run([PADDLE_PY,'-m','pip','install','paddleocr[doc-parser]==3.7.0','paddlex[ocr]==3.7.2','Pillow==11.3.0'],check=True)
vlm_versions={'torch':'2.6.0','torchvision':'0.21.0','transformers':'4.57.6','accelerate':'1.12.0','qwen-vl-utils':'0.0.14'}
if not versions_match(VLM_PY,vlm_versions):
    subprocess.run([VLM_PY,'-m','pip','install','torch==2.6.0','torchvision==0.21.0','--index-url','https://download.pytorch.org/whl/cu124/'],check=True)
    subprocess.run([VLM_PY,'-m','pip','install','transformers==4.57.6','accelerate==1.12.0','qwen-vl-utils==0.0.14','Pillow==11.3.0'],check=True)
if not versions_match(VLM_PY,{'bitsandbytes':'0.48.1'}):
    subprocess.run([VLM_PY,'-m','pip','install','bitsandbytes==0.48.1'],check=True)
if NEEDS_PADDLE:subprocess.run([PADDLE_PY,'-c','import paddle; paddle.utils.run_check(); assert paddle.is_compiled_with_cuda()'],check=True)
subprocess.run([VLM_PY,'-c','import torch,bitsandbytes; assert torch.cuda.is_available(); print(torch.ones(1,device="cuda"))'],check=True)
(OUT/('setup_seconds_'+str(time.time_ns())+'.json')).write_text(json.dumps({'seconds':time.perf_counter()-setup_start}))
for name,py in ([('paddle',PADDLE_PY)] if NEEDS_PADDLE else [])+[('vlm',VLM_PY)]:
    (OUT/(name+'_freeze_'+str(time.time_ns())+'.txt')).write_text(subprocess.check_output([py,'-m','pip','freeze'],text=True))


## Specialist reuse and revalidation
Reuse original inference with source hashes; recompute v3 quality flags. Failed identical formula crops stay in review instead of being blindly retried.


In [ ]:
def run(stage,python=sys.executable,extra=None):
    subprocess.run([python,str(WORK/'worker.py'),stage,'--input',str(WORK),'--out',str(OUT)]+(extra or []),check=True)
run('seed')
if NEEDS_PADDLE:run('specialists',PADDLE_PY)
run('collect')


## Pilot: four graphs plus two separate 4-Queens trees
Graph extraction is three stages. Every stage preserves raw output, elapsed time and independent cache identity.
A valid JSON/evidence gate never authorizes CPT export. Inspect arrow, weight and style evidence on the original crop.


In [ ]:
run('vlm',VLM_PY,['--scope','pilot'])
run('collect')
for r in MANIFEST['requests']:
    if r['engine']=='vlm' and r['page'] in MANIFEST['pilot_pages']:
        path=OUT/'records'/(r['id']+'.json')
        record=json.loads(path.read_text()) if path.exists() else {'status':'not_run'}
        print(r['id'],record['status'],record.get('quality',{}).get('flags',[]))


## Optional remaining VLM crops
Keep RUN_FULL=False until the pilot has been assessed. Setting it true only runs extraction; it does not verify output or append anything to CPT.
The worker requires a structurally successful pilot. Quality flags still require source review.


In [ ]:
RUN_FULL=False
if RUN_FULL:
    run('vlm',VLM_PY,['--scope','all'])
    run('collect')
else:
    print('Pilot only. Remaining VLM requests are honestly marked not_run.')


## Download snapshots, including errors
This cell can also be run after a failed pilot to download diagnostics. Earlier result snapshots remain intact.


In [ ]:
run('collect')
result_zip=BASE/('region_v3_results_'+RUN_HASH[:12]+'_'+str(time.time_ns())+'.zip')
with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,p.relative_to(OUT).as_posix())
    for p in [BASE/'setup_seconds.json',BASE/'paddle_freeze.txt',BASE/'vlm_freeze.txt']:
        if p.exists():z.write(p,p.name)
print(result_zip);files.download(str(result_zip))
